In [0]:
from pyspark.sql import functions as sf


df_bronze = (
    spark.read
    .format("text")
    .load("/Volumes/workspace/default/weatherproject/stations.txt")
)


df_bronze = (
    df_bronze
    .withColumn("filepath", sf.col("_metadata.file_path"))
    .withColumn("modificationtime", sf.col("_metadata.file_modification_time"))
    .withColumn("ingesttime", sf.current_timestamp())
)


df_bronze = (
    df_bronze
    .withColumn("ID", sf.trim(sf.substring("value", 1, 11)))
    .withColumn("LATITUDE", sf.trim(sf.substring("value", 13, 8)))
    .withColumn("LONGITUDE", sf.trim(sf.substring("value", 22, 9)))
    .withColumn("ELEVATION", sf.trim(sf.substring("value", 32, 6)))
    .withColumn("STATE", sf.trim(sf.substring("value", 39, 2)))
    .withColumn("NAME", sf.trim(sf.substring("value", 42, 30)))
    .withColumn("GSN_FLAG", sf.trim(sf.substring("value", 73, 3)))
    .withColumn("HCN_CRN_FLAG", sf.trim(sf.substring("value", 77, 3)))
    .withColumn("WMO_ID", sf.trim(sf.substring("value", 81, 5)))
)


df_bronze = df_bronze.select(
    "filepath",
    "modificationtime",
    "ingesttime",
    "ID",
    "LATITUDE",
    "LONGITUDE",
    "ELEVATION",
    "STATE",
    "NAME",
    "GSN_FLAG",
    "HCN_CRN_FLAG",
    "WMO_ID"
)


(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronzeweather.bronze_stations")
)